# Three-dimensional neural trajectories

This notebook independently fits and plots PC1-PC3 trajectories for the original combined, original separated, combined balanced, and separated balanced spatial models. Each model version receives its own PCA basis and three separate figures for chosen order, chosen juice, and chosen side.

By default the figures are displayed only. Set `SAVE_PDF = True` to save twelve independent PDF files; no PNG or GIF files are produced.

In [ ]:
from pathlib import Path
import re

from IPython.display import display
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
from sklearn.decomposition import PCA

mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42

ENSEMBLE_INDEX = 0
ACTIVITY_FILE = 'activitityTest.npz'
DT = 10
FIT_START_MS = 500
FIT_END_MS = 3200
SAVE_PDF = False

MODEL_DIR_OVERRIDES = {
    'combined': None,
    'seperated': None,
    'balanced_combined': None,
    'balanced_separated': None,
}

PHASE_BOUNDARIES_MS = [500, 1000, 1500, 2000, 2500, 3000, 3200]
VIEW_ELEVATION = 22
VIEW_AZIMUTH = -58

In [ ]:
def find_analysis_dir():
    candidates = [
        Path.cwd(),
        Path.cwd() / 'Analysis',
        Path.cwd() / 'EconomicDecisionMakingReferenceFrame' / 'Analysis',
    ]
    for candidate in candidates:
        if (candidate / 'plot_neural_trajectories_3d.ipynb').exists():
            return candidate.resolve()
    raise FileNotFoundError(
        'Run this notebook from the repository root or Analysis directory.')


def select_model_dir(task_dir, ensemble_index, override=None):
    if override is not None:
        model_dir = Path(override).expanduser().resolve()
        if not (model_dir / ACTIVITY_FILE).exists():
            raise FileNotFoundError(f'Activity file not found in override: {model_dir}')
        return model_dir

    if not task_dir.exists():
        raise FileNotFoundError(f'Model collection not found: {task_dir}')
    pattern = re.compile(rf'_{ensemble_index}_(?:Fail)?$')
    candidates = [
        path for path in task_dir.iterdir()
        if path.is_dir()
        and pattern.search(path.name)
        and (path / ACTIVITY_FILE).exists()
    ]
    if not candidates:
        raise FileNotFoundError(
            f'No ensemble {ensemble_index} with {ACTIVITY_FILE} under {task_dir}.')

    successful = [path for path in candidates if not path.name.endswith('_Fail')]
    pool = successful if successful else candidates
    selected = sorted(pool, key=lambda path: path.name)[-1]
    if len(candidates) > 1:
        print(f'Warning: found {len(candidates)} candidates; using newest: {selected.name}')
    return selected


def loc12_to_label(params):
    value = params.get('loc12_label', params.get('loc12'))
    if isinstance(value, str):
        return value
    value = np.asarray(value).reshape(-1)
    return '12' if value[0] == 1 else '21'


def mean_response_output(model_output, trial_params, mask, dt):
    if mask is not None and np.any(mask):
        denominator = np.sum(mask, axis=1)
        denominator[denominator == 0] = 1
        return np.sum(mask * model_output, axis=1) / denominator

    response = np.zeros((model_output.shape[0], model_output.shape[2]))
    for trial_index, params in enumerate(trial_params):
        response_onset = params.get('fixation_offset', 3000)
        response_offset = params.get('end', 3200)
        start = max(0, int(round(response_onset / dt)))
        stop = min(model_output.shape[1], int(round(response_offset / dt)))
        stop = max(start + 1, stop)
        response[trial_index] = model_output[trial_index, start:stop].mean(axis=0)
    return response


def load_activity_and_labels(model_dir, activity_file=ACTIVITY_FILE, dt=DT):
    with np.load(model_dir / activity_file, allow_pickle=True) as data:
        model_state = data['model_state']
        model_output = data['model_output']
        trial_params = data['trial_params']
        mask = data['mask'] if 'mask' in data.files else None

    response = mean_response_output(model_output, trial_params, mask, dt)
    choice_side = np.where(response[:, 1] > response[:, 0], 'right', 'left')
    seq_ab = np.asarray([params['seqAB'] for params in trial_params])
    loc12 = np.asarray([loc12_to_label(params) for params in trial_params])

    offer1_left = loc12 == '12'
    chose_offer1 = ((choice_side == 'left') & offer1_left) | (
        (choice_side == 'right') & ~offer1_left)
    choice_order = np.where(chose_offer1, '1', '2')
    choice_juice = np.where(
        ((seq_ab == 'AB') & (choice_order == '1'))
        | ((seq_ab == 'BA') & (choice_order == '2')),
        'A',
        'B')

    labels = {
        'seqAB': seq_ab,
        'chosen order': choice_order,
        'chosen juice': choice_juice,
        'chosen side': choice_side,
    }
    return model_state, labels

In [ ]:
def fit_and_project_states(model_state, start_ms, end_ms, dt):
    n_trials, n_time, n_units = model_state.shape
    start = max(0, int(round(start_ms / dt)))
    stop = min(n_time, int(round(end_ms / dt)) + 1)
    if stop - start < 2:
        raise ValueError(f'Invalid PCA window: {start_ms}-{end_ms} ms')

    states = model_state[:, start:stop, :]
    pca = PCA(n_components=3)
    projected = pca.fit_transform(states.reshape(-1, n_units))
    projected = projected.reshape(n_trials, stop - start, 3)
    time = np.arange(start, stop) * dt
    return pca, projected, time


def trajectory_groups(labels):
    seq_ab = labels['seqAB']
    choice_order = labels['chosen order']
    choice_juice = labels['chosen juice']
    choice_side = labels['chosen side']

    return [
        (
            'chosen_order',
            'Chosen order',
            [
                ('AB / choose 1', (seq_ab == 'AB') & (choice_order == '1'), 'tab:orange', '-'),
                ('AB / choose 2', (seq_ab == 'AB') & (choice_order == '2'), 'tomato', '--'),
                ('BA / choose 1', (seq_ab == 'BA') & (choice_order == '1'), 'tab:blue', '-'),
                ('BA / choose 2', (seq_ab == 'BA') & (choice_order == '2'), 'deepskyblue', '--'),
            ],
        ),
        (
            'chosen_juice',
            'Chosen juice',
            [
                ('choose A', choice_juice == 'A', 'tab:orange', '-'),
                ('choose B', choice_juice == 'B', 'tab:blue', '-'),
            ],
        ),
        (
            'chosen_side',
            'Chosen side',
            [
                ('choose left', choice_side == 'left', 'tab:green', '-'),
                ('choose right', choice_side == 'right', 'tab:purple', '-'),
            ],
        ),
    ]


def set_3d_limits(ax, trajectories):
    flat = trajectories.reshape(-1, 3)
    lower = np.nanmin(flat, axis=0)
    upper = np.nanmax(flat, axis=0)
    span = np.maximum(upper - lower, 1e-6)
    padding = span * 0.08
    lower -= padding
    upper += padding
    ax.set_xlim(lower[0], upper[0])
    ax.set_ylim(lower[1], upper[1])
    ax.set_zlim(lower[2], upper[2])
    ax.set_box_aspect((1.35, 1, 1))


def plot_model_trajectories_3d(model_name, model_dir, model_state, labels):
    pca, projected, time = fit_and_project_states(
        model_state, FIT_START_MS, FIT_END_MS, DT)
    variance = pca.explained_variance_ratio_

    boundary_indices = [
        int(np.argmin(np.abs(time - boundary)))
        for boundary in PHASE_BOUNDARIES_MS
        if time[0] <= boundary <= time[-1]
    ]

    print(f'\n{model_name}: {model_dir}')
    print('PCA explained variance:', np.round(variance, 4),
          'cumulative:', round(float(variance.sum()), 4))

    model_figures = {}
    for panel_key, panel_title, groups in trajectory_groups(labels):
        fig = plt.figure(figsize=(8.5, 6.5), dpi=150, constrained_layout=True)
        ax = fig.add_subplot(1, 1, 1, projection='3d')
        panel_trajectories = []
        for group_label, mask, color, linestyle in groups:
            count = int(np.sum(mask))
            if count == 0:
                raise ValueError(f'{model_name}: empty group {group_label}')
            trace = projected[mask].mean(axis=0)
            panel_trajectories.append(trace)
            ax.plot(trace[:, 0], trace[:, 1], trace[:, 2],
                    color=color, linestyle=linestyle, linewidth=2.3,
                    label=f'{group_label} (n={count})')
            ax.scatter(trace[boundary_indices, 0], trace[boundary_indices, 1],
                       trace[boundary_indices, 2], color=color, s=18, alpha=.8)
            ax.scatter(trace[0, 0], trace[0, 1], trace[0, 2],
                       marker='X', color='black', edgecolors='white',
                       linewidths=1.2, s=120, depthshade=False)
            ax.scatter(trace[-1, 0], trace[-1, 1], trace[-1, 2],
                       color=color, s=26, depthshade=False)

        set_3d_limits(ax, np.stack(panel_trajectories))
        ax.set_xlabel(f'PC1 ({variance[0] * 100:.1f}%)', labelpad=6)
        ax.set_ylabel(f'PC2 ({variance[1] * 100:.1f}%)', labelpad=6)
        ax.set_zlabel(f'PC3 ({variance[2] * 100:.1f}%)', labelpad=6)
        ax.view_init(elev=VIEW_ELEVATION, azim=VIEW_AZIMUTH)
        ax.legend(frameon=False, fontsize=8, loc='upper left',
                  bbox_to_anchor=(1.01, 1.0), borderaxespad=0)
        fig.suptitle(
            f'{model_name}: {panel_title} | PC1-PC3 neural trajectory\n'
            f'cumulative variance = {variance.sum() * 100:.1f}% | '
            f'{model_dir.name}',
            fontsize=11)
        model_figures[panel_key] = fig

    return model_figures, pca

In [ ]:
analysis_dir = find_analysis_dir()
saved_local = analysis_dir.parent / 'Training' / 'savedForLocal'
figure_dir = analysis_dir / 'Figure'

balanced_combined_candidates = [
    saved_local / 'spatialTask_balanced_combined',
    saved_local / 'spatialTaskCombinedBalanced',
    saved_local / 'balanced_combined',
]
balanced_combined_root = next(
    (candidate for candidate in balanced_combined_candidates if candidate.exists()),
    balanced_combined_candidates[0])

model_specs = {
    'combined': saved_local / 'spatialTask_combined',
    'seperated': saved_local / 'spatialTask_seperated',
    'balanced_combined': balanced_combined_root,
    'balanced_separated': saved_local / 'spatialTask_balanced_separated',
}

figures = {}
pca_objects = {}
for model_name, task_dir in model_specs.items():
    try:
        model_dir = select_model_dir(
            task_dir, ENSEMBLE_INDEX, MODEL_DIR_OVERRIDES[model_name])
        model_state, labels = load_activity_and_labels(model_dir)
        model_figures, pca = plot_model_trajectories_3d(
            model_name, model_dir, model_state, labels)
        figures[model_name] = model_figures
        pca_objects[model_name] = pca

        for panel_key, fig in model_figures.items():
            if SAVE_PDF:
                figure_dir.mkdir(parents=True, exist_ok=True)
                output_path = figure_dir / (
                    f'neural_trajectories_3d_{model_name}_{panel_key}.pdf')
                fig.savefig(output_path, format='pdf', bbox_inches='tight')
                print('Saved:', output_path)

            display(fig)
            plt.close(fig)
    except FileNotFoundError as error:
        print(f'Skipped {model_name}: {error}')

if not figures:
    raise FileNotFoundError('No model version could be loaded from savedForLocal.')